## 准备数据

In [1]:
import os
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, optimizers, datasets

os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'  # or any {'0', '1', '2'}

def mnist_dataset():
    (x, y), (x_test, y_test) = datasets.mnist.load_data()
    #normalize
    x = x/255.0
    x_test = x_test/255.0
    
    return (x, y), (x_test, y_test)

In [2]:
print(list(zip([1, 2, 3, 4], ['a', 'b', 'c', 'd'])))

[(1, 'a'), (2, 'b'), (3, 'c'), (4, 'd')]


## 建立模型

In [3]:
class myModel:
    def __init__(self):
        ####################
        '''声明模型对应的参数'''
        ####################
        self.W1 = tf.Variable(tf.random.normal([28 * 28, 100], stddev=0.1))
        self.b1 = tf.Variable(tf.zeros([100]))
        self.W2 = tf.Variable(tf.random.normal([100, 10], stddev=0.1))
        self.b2 = tf.Variable(tf.zeros([10]))
    def __call__(self, x):
        ####################
        '''实现模型函数体，返回未归一化的logits'''
        ####################
        x = tf.reshape(x, [-1, 28 * 28])
        h1 = tf.nn.relu(tf.matmul(x, self.W1) + self.b1)
        logits = tf.matmul(h1, self.W2) + self.b2
        return logits
        
model = myModel()

optimizer = optimizers.Adam()

## 计算 loss

In [4]:
@tf.function
def compute_loss(logits, labels):
    return tf.reduce_mean(
        tf.nn.sparse_softmax_cross_entropy_with_logits(
            logits=logits, labels=labels))

@tf.function
def compute_accuracy(logits, labels):
    predictions = tf.argmax(logits, axis=1)
    return tf.reduce_mean(tf.cast(tf.equal(predictions, labels), tf.float32))

@tf.function
def train_one_step(model, optimizer, x, y):
    with tf.GradientTape() as tape:
        logits = model(x)
        loss = compute_loss(logits, y)

    # compute gradient
    trainable_vars = [model.W1, model.W2, model.b1, model.b2]
    grads = tape.gradient(loss, trainable_vars)
    for g, v in zip(grads, trainable_vars):
        v.assign_sub(0.01*g)

    accuracy = compute_accuracy(logits, y)

    # loss and accuracy is scalar tensor
    return loss, accuracy

@tf.function
def test(model, x, y):
    logits = model(x)
    loss = compute_loss(logits, y)
    accuracy = compute_accuracy(logits, y)
    return loss, accuracy

## 实际训练

In [5]:
train_data, test_data = mnist_dataset()
for epoch in range(50):
    loss, accuracy = train_one_step(model, optimizer, 
                                    tf.constant(train_data[0], dtype=tf.float32), 
                                    tf.constant(train_data[1], dtype=tf.int64))
    print('epoch', epoch, ': loss', loss.numpy(), '; accuracy', accuracy.numpy())
loss, accuracy = test(model, 
                      tf.constant(test_data[0], dtype=tf.float32), 
                      tf.constant(test_data[1], dtype=tf.int64))

print('test loss', loss.numpy(), '; accuracy', accuracy.numpy())

epoch 0 : loss 2.5219982 ; accuracy 0.0998
epoch 1 : loss 2.4961145 ; accuracy 0.0996
epoch 2 : loss 2.4727495 ; accuracy 0.09955
epoch 3 : loss 2.451485 ; accuracy 0.09993333
epoch 4 : loss 2.431989 ; accuracy 0.09956667
epoch 5 : loss 2.413986 ; accuracy 0.1003
epoch 6 : loss 2.3972538 ; accuracy 0.10136667
epoch 7 : loss 2.381615 ; accuracy 0.10235
epoch 8 : loss 2.3669214 ; accuracy 0.10425
epoch 9 : loss 2.3530543 ; accuracy 0.10571667
epoch 10 : loss 2.339911 ; accuracy 0.1072
epoch 11 : loss 2.3274045 ; accuracy 0.108833335
epoch 12 : loss 2.3154614 ; accuracy 0.11068334
epoch 13 : loss 2.3040206 ; accuracy 0.11338333
epoch 14 : loss 2.2930303 ; accuracy 0.11618333
epoch 15 : loss 2.2824442 ; accuracy 0.1198
epoch 16 : loss 2.2722204 ; accuracy 0.12336667
epoch 17 : loss 2.2623253 ; accuracy 0.1277
epoch 18 : loss 2.2527282 ; accuracy 0.132
epoch 19 : loss 2.2434027 ; accuracy 0.13666667
epoch 20 : loss 2.2343242 ; accuracy 0.14226666
epoch 21 : loss 2.2254753 ; accuracy 0.14756